# 04 · Forward propagation

## What you will build

The complete forward pass of your network: a numerically stable softmax, a `forward` function that runs every layer and keeps a **cache** of what each layer saw (backpropagation will need it), and `predict` and `accuracy` on top. You will also check that a batched forward pass gives exactly the same answers as one example at a time.

## Learning objectives

- Trace the shape of the data through every layer of a network.
- Implement a stable, row-wise softmax.
- Implement `forward(network, X)` returning probabilities and a cache.
- Explain why the batched forward pass is equivalent to a per-example loop, and why it is faster.

## Dependencies

Tier 1: NumPy and Matplotlib only.

## Setup

Run this cell first. It loads NumPy and Matplotlib and defines `practice`, the checker used after every ✍️ cell.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import numpy as np

rng = np.random.default_rng(0)


def _close(got, want, atol):
    if isinstance(want, dict):
        return isinstance(got, dict) and got.keys() == want.keys() and all(
            _close(got[key], want[key], atol) for key in want
        )
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want)
        )
    try:
        got_array, want_array = np.asarray(got, dtype=float), np.asarray(want, dtype=float)
    except (TypeError, ValueError):
        return got == want
    return got_array.shape == want_array.shape and np.allclose(got_array, want_array, atol=atol)


def _short(value):
    if isinstance(value, tuple):
        return "(" + ", ".join(_short(item) for item in value) + ")"
    if isinstance(value, np.ndarray):
        text = np.array2string(value, precision=4, threshold=8, separator=", ").replace("\n", "")
    else:
        text = repr(value)
    return text if len(text) < 120 else text[:117] + "..."


def practice(name, yours, reference, cases, atol=1e-6):
    """Check your function against the reference on a few inputs.

    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference -- so you can always keep going.
    """
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got)}")
                print(f"   expected     {_short(want)}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: matches the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. From here on the notebook runs YOUR {name}.")
    return yours

import matplotlib.pyplot as plt

## 📦 Recap: what you built earlier

The hidden cell below defines the pieces from earlier notebooks under their usual names, so this notebook runs on its own. Open it if you want to see them again.

In [ ]:
# 📦 Recap: the pieces from notebook 01, under their usual names.

def make_spirals(n_per_class, n_classes, noise, rng):
    """n_classes interleaved spiral arms in 2-D, sorted by class: X (n, 2), integer labels y (n,)."""
    radius = np.linspace(0.05, 1.0, n_per_class)
    X_parts, y_parts = [], []
    for k in range(n_classes):
        angle = 2 * np.pi * k / n_classes + 6.0 * radius + rng.normal(0.0, noise, n_per_class)
        X_parts.append(np.column_stack([radius * np.cos(angle), radius * np.sin(angle)]))
        y_parts.append(np.full(n_per_class, k))
    return np.concatenate(X_parts), np.concatenate(y_parts)


# From notebooks 02 and 03.

def relu(Z):
    return np.maximum(0.0, Z)


def relu_grad(Z):
    return (Z > 0).astype(float)


def dense_forward(X, W, b):
    return X @ W + b


def init_network(sizes, rng):
    return [{"W": rng.normal(0.0, np.sqrt(2.0 / n_in), (n_in, n_out)), "b": np.zeros(n_out)}
            for n_in, n_out in zip(sizes[:-1], sizes[1:])]

## 1 · Follow the shapes

For a `[2, 16, 16, 3]` network and a batch of 5 examples, each layer turns `(5, n_in)` into `(5, n_out)`. The batch size stays in the first axis; the feature axis changes.

In [ ]:
sizes = [2, 16, 16, 3]
demo_net = init_network(sizes, np.random.default_rng(0))
A = np.random.default_rng(1).normal(size=(5, 2))
print(f"input            {A.shape}")
for i, layer in enumerate(demo_net):
    Z = dense_forward(A, layer["W"], layer["b"])
    A = Z if i == len(demo_net) - 1 else relu(Z)
    print(f"layer {i}: W{layer['W'].shape} -> Z{Z.shape}")

## 2 · Softmax: from scores to probabilities

The last layer produces one score (a *logit*) per class. Softmax turns a row of logits into probabilities that are positive and sum to 1:

$$\operatorname{softmax}(z)_k = \frac{e^{z_k}}{\sum_j e^{z_j}}.$$

Large logits overflow `np.exp`. Subtracting the row's maximum first changes nothing mathematically (the factor $e^{-\max}$ cancels) and keeps every exponent at most 0.

### ✍️ Your turn: `softmax`

Return the row-wise softmax of `Z` (shape `(batch, classes)`): subtract each row's maximum (`keepdims=True`), exponentiate, divide by the row sum.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below if you get stuck.

In [ ]:
def softmax_reference(Z):
    exps = np.exp(Z - Z.max(axis=1, keepdims=True))
    return exps / exps.sum(axis=1, keepdims=True)

In [ ]:
def softmax(Z):
    """Row-wise, numerically stable softmax."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
softmax = practice("softmax", softmax, softmax_reference, cases=[
    (np.array([[1.0, 2.0, 3.0], [0.0, 0.0, 0.0]]),),
    (np.array([[1000.0, 1001.0], [-1000.0, -999.0]]),),
])

## 3 · The forward pass, with a cache

`forward` runs each layer in order: ReLU after every hidden layer, softmax after the last. For each layer it stores the pair `(A_prev, Z)`, the layer's input and its pre-activation output. Backpropagation (notebook 07) reuses exactly these arrays instead of recomputing them.

### ✍️ Your turn: `forward`

Return `(probs, cache)`. Start with `A = X`; for layer `i`, compute `Z = dense_forward(A, W, b)`, append `(A, Z)` to `cache`, then set `A = softmax(Z)` for the last layer and `relu(Z)` otherwise. `probs` is the final `A`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below if you get stuck.

In [ ]:
def forward_reference(network, X):
    cache, A = [], X
    for index, layer in enumerate(network):
        Z = dense_forward(A, layer["W"], layer["b"])
        cache.append((A, Z))
        A = softmax_reference(Z) if index == len(network) - 1 else relu(Z)
    return A, cache

In [ ]:
def forward(network, X):
    """ReLU hidden layers, softmax output. Returns (probs, cache)."""
    cache = []
    A = X
    # ✍️ Your code here: one loop over enumerate(network)
    raise NotImplementedError

In [ ]:
X_check = np.random.default_rng(4).normal(size=(6, 2))
forward = practice("forward", forward, forward_reference, cases=[
    (demo_net, X_check),
    (init_network([2, 5, 3], np.random.default_rng(5)), X_check[:2]),
])

### ✍️ Your turn: `predict`

Return the predicted class for every row of `X`: the index of the largest probability, `probs.argmax(axis=1)`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below if you get stuck.

In [ ]:
def predict_reference(network, X):
    return forward(network, X)[0].argmax(axis=1)

In [ ]:
def predict(network, X):
    """Predicted class indices, shape (batch,)."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
predict = practice("predict", predict, predict_reference, cases=[(demo_net, X_check)])

### ✍️ Your turn: `accuracy`

Return the fraction of rows whose predicted class equals `y`, as a float.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below if you get stuck.

In [ ]:
def accuracy_reference(network, X, y):
    return float(np.mean(predict(network, X) == y))

In [ ]:
def accuracy(network, X, y):
    """Fraction of correct predictions."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
accuracy = practice("accuracy", accuracy, accuracy_reference, cases=[
    (demo_net, X_check, np.array([0, 1, 2, 0, 1, 2])),
])

## 4 · A batch equals a loop, only faster

Each row of a batch is processed independently, so running the whole batch at once must give the same probabilities as running each example on its own.

In [ ]:
X, y = make_spirals(200, 3, 0.2, np.random.default_rng(1))
network = init_network([2, 64, 3], np.random.default_rng(2))
batched, _ = forward(network, X)
looped = np.vstack([forward(network, X[i:i + 1])[0] for i in range(len(X))])
print("largest difference between batched and looped:", np.abs(batched - looped).max())

### 🧪 Try it: how much faster is the batch?

In [ ]:
import time
for size in (1, 32, 600):
    start = time.perf_counter()
    for row in range(0, len(X), size):
        forward(network, X[row:row + size])
    print(f"batch size {size:4d}: {1000 * (time.perf_counter() - start):6.1f} ms for {len(X)} examples")

## 5 · Put it together

In [ ]:
probs, cache = forward(network, X)
assert probs.shape == (len(X), 3)
assert np.allclose(probs.sum(axis=1), 1.0), "every row of probabilities must sum to 1"
assert len(cache) == len(network)
assert cache[0][0] is X and cache[1][1].shape == (len(X), 3)
assert np.allclose(batched, looped)
untrained = accuracy(network, X, y)
assert untrained < 0.6
print(f"✅ Forward pass complete. Untrained accuracy {untrained:.2f}: next, measure how wrong it is.")

## Exercises

1. **Logits too.** Change a copy of `forward` so it also returns the last layer's logits. Why might a loss computed from logits be more stable than one computed from probabilities? (Notebook 05.)
2. **Temperature.** Compute `softmax(Z / T)` for $T = 0.5, 1, 2$ on one row. What happens to the probabilities as $T$ grows?
3. **Memory.** For a batch of 1,000 and `[2, 512, 512, 10]`, how many numbers does the cache hold? Why does training memory grow with batch size?
4. **Open question.** `forward` keeps `A_prev` and `Z` for every layer. Which of them will the backward pass need, and for what?

## Common errors and debugging

- **Probabilities that do not sum to 1**: you used `axis=0` (columns) instead of `axis=1` (rows).
- **`nan` in the output**: large logits overflowed; subtract the row maximum.
- **Wrong cache contents**: append `(A, Z)` *before* overwriting `A` with the activation.

## Summary

The forward pass is a loop of matrix products and activations, ending in a softmax that turns scores into probabilities. Keeping a cache of each layer's input and pre-activation makes the backward pass cheap. Processing a batch gives the same result as a loop, much faster.

## Next

**05 · Loss functions**: turn "how wrong" into a single number the network can reduce.